# Rental Price Prediction — Model Development Notebook

**Goal:** Predict monthly apartment rental price (`price`) from property characteristics and location.

**Workflow:**
1. Load & explore data
2. Preprocessing pipeline
3. Validation strategy (5-fold CV)
4. Baseline model comparison (Linear Regression, Decision Tree, Random Forest, Gradient Boosting)
5. Overfitting/underfitting diagnostics
6. Hyperparameter tuning (Grid Search)
7. Feature engineering
8. Final model selection & justification
9. Train final model & generate `predictions.csv`

> Fill in the markdown "TODO" cells with your team's own analysis/discussion before submitting.

## 0. Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, KFold, cross_val_score, GridSearchCV
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_squared_error

from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor

RANDOM_STATE = 42
pd.set_option("display.max_columns", None)


## 1. Load the data

Place `challenge1_train.csv` and `challenge1_test.csv` in the same folder as this notebook (or update the paths below).

In [ ]:
train_df = pd.read_csv("challenge1_train.csv")
test_df = pd.read_csv("challenge1_test.csv")

print("Train shape:", train_df.shape)
print("Test shape:", test_df.shape)
train_df.head()


## 2. Explore the data

In [ ]:
train_df.info()


In [ ]:
# Missing values per column
missing = train_df.isna().sum()
missing = missing[missing > 0].sort_values(ascending=False)
print("Columns with missing values:")
print(missing if len(missing) else "None")


In [ ]:
# Summary statistics for numeric columns
train_df.describe()


In [ ]:
# Target variable distribution
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(train_df["price"], bins=50)
axes[0].set_title("Price distribution")
axes[0].set_xlabel("price")

axes[1].hist(np.log1p(train_df["price"]), bins=50)
axes[1].set_title("log1p(price) distribution")
axes[1].set_xlabel("log1p(price)")
plt.tight_layout()
plt.show()

print("Skew of price:", train_df["price"].skew())
print("Skew of log1p(price):", np.log1p(train_df["price"]).skew())


**TODO (team):** Note anything you observe here — skewness, outliers, categorical cardinality, columns that look like they need cleaning/engineering, etc. If `price` is strongly right-skewed, consider modeling `log1p(price)` and inverting with `expm1()` at prediction time (see the optional log-target section near the end).

## 3. Separate features/target and identify column types

In [ ]:
# ID should never be used as a predictor
X = train_df.drop(columns=["price", "ID"], errors="ignore")
y = train_df["price"]

test_ids = test_df["ID"]
X_test_final = test_df.drop(columns=["ID"], errors="ignore")

# Automatically split columns by dtype. Double check this matches what you'd
# expect after your EDA above (e.g. a column that LOOKS numeric but is really
# a categorical code should be moved to categorical_cols manually).
numeric_cols = X.select_dtypes(include=["int64", "float64"]).columns.tolist()
categorical_cols = X.select_dtypes(include=["object", "category", "bool"]).columns.tolist()

print("Numeric columns:", numeric_cols)
print("Categorical columns:", categorical_cols)


## 4. Validation strategy

We use two complementary checks, both driven by the **same 5-fold split** so results are comparable across every model and tuning step:

- **5-fold cross-validation RMSE** on the full training set — the primary number used to compare and select models.
- **A single 80/20 hold-out split** — used only to compute a *train* RMSE vs. a *validation* RMSE for each model, which tells us whether a model is overfitting (train RMSE much lower than validation RMSE) or underfitting (both high and close together).

In [ ]:
cv = KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

X_tr, X_val, y_tr, y_val = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE
)

def rmse(y_true, y_pred):
    return np.sqrt(mean_squared_error(y_true, y_pred))


## 5. Preprocessing pipeline

In [ ]:
preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numeric_cols),
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_cols),
    ]
)


## 6. Define the required models

Linear Regression, Decision Tree, Random Forest, and Gradient Boosting, each wrapped in the same preprocessing pipeline so every model sees identical inputs.

In [ ]:
models = {
    "Linear Regression": LinearRegression(),
    "Decision Tree": DecisionTreeRegressor(random_state=RANDOM_STATE),
    "Random Forest": RandomForestRegressor(random_state=RANDOM_STATE, n_jobs=-1),
    "Gradient Boosting": GradientBoostingRegressor(random_state=RANDOM_STATE),
}

pipelines = {
    name: Pipeline([("preprocess", preprocessor), ("model", model)])
    for name, model in models.items()
}


## 7. Cross-validate and compare models

For each model:
- `CV RMSE` = mean RMSE across the 5 folds on the full training set (primary comparison metric).
- `Train RMSE` / `Val RMSE` = fit on the 80% hold-out train split, scored on train and val respectively (overfitting/underfitting check).

In [ ]:
results = []

for name, pipe in pipelines.items():
    # 5-fold CV RMSE on the full training data
    cv_scores = cross_val_score(
        pipe, X, y, scoring="neg_root_mean_squared_error", cv=cv, n_jobs=-1
    )
    cv_rmse_mean = -cv_scores.mean()
    cv_rmse_std = cv_scores.std()

    # Hold-out split: fit on train, score on train and val to check over/underfitting
    pipe.fit(X_tr, y_tr)
    train_rmse = rmse(y_tr, pipe.predict(X_tr))
    val_rmse = rmse(y_val, pipe.predict(X_val))

    results.append({
        "Model": name,
        "CV RMSE (mean)": cv_rmse_mean,
        "CV RMSE (std)": cv_rmse_std,
        "Train RMSE": train_rmse,
        "Val RMSE": val_rmse,
        "Gap (Val - Train)": val_rmse - train_rmse,
    })

results_df = pd.DataFrame(results).sort_values("CV RMSE (mean)").reset_index(drop=True)
results_df


**TODO (team):** Interpret this table.
- Which model has the lowest CV RMSE?
- Which model(s) show a large `Gap (Val - Train)` (a sign of overfitting)? Decision trees and, to a lesser extent, random forests often overfit if left untuned.
- Which model(s) show high Train AND Val RMSE close together (a sign of underfitting)? A plain linear regression often falls here if the relationship is non-linear.

## 8. Hyperparameter tuning with Grid Search

We tune the two ensemble models, since they typically offer the most room for improvement over their defaults. Feel free to also tune the Decision Tree the same way (a small grid over `max_depth` / `min_samples_leaf` usually fixes most of its overfitting).

In [ ]:
rf_param_grid = {
    "model__n_estimators": [200, 400],
    "model__max_depth": [None, 10, 20],
    "model__min_samples_leaf": [1, 2, 4],
}

rf_grid = GridSearchCV(
    pipelines["Random Forest"],
    param_grid=rf_param_grid,
    scoring="neg_root_mean_squared_error",
    cv=cv,
    n_jobs=-1,
    verbose=1,
)
rf_grid.fit(X, y)

print("Best Random Forest params:", rf_grid.best_params_)
print("Best Random Forest CV RMSE:", -rf_grid.best_score_)


In [ ]:
gb_param_grid = {
    "model__n_estimators": [100, 200, 300],
    "model__learning_rate": [0.01, 0.05, 0.1],
    "model__max_depth": [2, 3, 4],
}

gb_grid = GridSearchCV(
    pipelines["Gradient Boosting"],
    param_grid=gb_param_grid,
    scoring="neg_root_mean_squared_error",
    cv=cv,
    n_jobs=-1,
    verbose=1,
)
gb_grid.fit(X, y)

print("Best Gradient Boosting params:", gb_grid.best_params_)
print("Best Gradient Boosting CV RMSE:", -gb_grid.best_score_)


In [ ]:
tuned_results = pd.DataFrame([
    {"Model": "Random Forest (tuned)", "CV RMSE": -rf_grid.best_score_},
    {"Model": "Gradient Boosting (tuned)", "CV RMSE": -gb_grid.best_score_},
])

comparison = pd.concat([
    results_df[["Model", "CV RMSE (mean)"]].rename(columns={"CV RMSE (mean)": "CV RMSE"}),
    tuned_results
], ignore_index=True).sort_values("CV RMSE").reset_index(drop=True)

comparison


## 9. Feature engineering (optional but encouraged)

Ideas to try — adapt these to the columns your dataset actually has (check the column list you printed in section 3):

- **Ratios/derived numeric features**, e.g. `price_per_sqft`-style ratios *computed only from other predictors, never from the target*, rooms-per-area, etc.
- **Binning** a skewed numeric column (e.g. distance-to-center) into categories.
- **Interaction terms** between two features you suspect interact (e.g. neighborhood × property type).
- **Log-transforming** heavily skewed numeric predictors (check with `.skew()` like we did for `price`).
- **Grouping rare categories**: if a categorical column has many rare levels, group everything below a frequency threshold into an `"Other"` bucket before one-hot encoding — this reduces the dimensionality `OneHotEncoder` produces and can reduce overfitting.

After adding any engineered features to `X` (and the matching transformation to `X_test_final`), re-run the cross-validation cell in section 7 to see whether CV RMSE improves. Keep a feature only if it measurably helps.

In [ ]:
# Example: grouping rare categories in a categorical column before one-hot encoding.
# Uncomment and adapt to a real column name from categorical_cols.

# def group_rare_categories(df, column, threshold=0.01):
#     freq = df[column].value_counts(normalize=True)
#     rare = freq[freq < threshold].index
#     return df[column].where(~df[column].isin(rare), "Other")
#
# X["some_categorical_col"] = group_rare_categories(X, "some_categorical_col")
# X_test_final["some_categorical_col"] = group_rare_categories(X_test_final, "some_categorical_col")


## 10. Final model selection

**TODO (team):** State which model you are selecting as final, and justify it referencing:
- Validation performance (CV RMSE from the comparison table above)
- Overfitting/underfitting (the Train/Val RMSE gap from section 7)
- Model complexity
- Computational cost (training/inference time)
- Interpretability

Update the variable `FINAL_PIPELINE` in the next code cell to point at whichever pipeline your team selects (e.g. `rf_grid.best_estimator_`, `gb_grid.best_estimator_`, or a plain `pipelines["Linear Regression"]`).

In [ ]:
# Set this to the pipeline your team selected as final, e.g.:
# FINAL_PIPELINE = gb_grid.best_estimator_
# FINAL_PIPELINE = rf_grid.best_estimator_

FINAL_PIPELINE = gb_grid.best_estimator_  # <-- update if your team picks a different model


## 11. Train the final model on the full training set and predict on the test set

`GridSearchCV` with `refit=True` (the default) already refits the best estimator on the *entire* training set it was given, so if `FINAL_PIPELINE` came from `rf_grid.best_estimator_` or `gb_grid.best_estimator_`, it's already fit on all of `X`/`y`. The explicit `.fit(X, y)` below is included so this cell works correctly regardless of which pipeline you chose.

In [ ]:
FINAL_PIPELINE.fit(X, y)

test_predictions = FINAL_PIPELINE.predict(X_test_final)

predictions_df = pd.DataFrame({
    "ID": test_ids,
    "price": test_predictions
})

predictions_df.to_csv("predictions.csv", index=False)
predictions_df.head()


## 12. (Optional) Modeling `log1p(price)` instead of `price`

If section 2 showed `price` is strongly right-skewed, retrying the comparison in section 7 with `y_log = np.log1p(y)` as the target (and inverting predictions with `np.expm1(...)` before computing RMSE / writing `predictions.csv`) sometimes improves RMSE on the original scale. This is worth trying and comparing against the untransformed results if you have time.